In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision', 'safetensors', 'tiktoken']
imports = {'pillow': 'PIL'}
pinned = {'safetensors': ('0.8.0', 'safetensors==0.8.0', 'safetensors==0.8.0', 'exact'), 'tiktoken': ('0.13.0', 'tiktoken==0.13.0', 'tiktoken==0.13.0', 'exact')}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Decoder-Only Language Models (GPT)

A single `TransformerBlock` (that section) maps a sequence of $d$-dimensional token representations to a sequence of the same shape $(B, T, d)$. To turn a stack of causal Transformer blocks into an autoregressive **Generative Pre-trained Transformer (GPT)** [@Radford.Narasimhan.Salimans.ea.2018; @Radford.Wu.Child.ea.2019; @brown2020language], we wrap $L$ blocks between two boundary layers (the figure):

1. **Input embedding:** A token embedding table $\mathbf{E} \in \mathbb{R}^{V \times d}$ maps discrete token IDs $x_1, \dots, x_T \in \{0, \dots, V-1\}$ to vectors in $\mathbb{R}^d$ (optionally adding learned absolute position embeddings $\mathbf{P} \in \mathbb{R}^{T_{\max} \times d}$ when rotary position embeddings are not used inside attention).
2. **Final normalization and language-modeling head:** After $L$ causal `TransformerBlock` layers, a final normalization layer stabilizes the residual stream $\mathbf{H}^{(L)}$ and a linear head projects each position $t$ to next-token logits $\mathbf{Z} = \textrm{Norm}(\mathbf{H}^{(L)})\mathbf{E}^\top \in \mathbb{R}^{B \times T \times V}$, **tying** the output projection weights to the input embedding table $\mathbf{E}$ [@Press.Wolf.2017; @Inan.Khosravi.Socher.2017].

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## From Blocks to a Language Model

![A GPT from bottom to top. A table $\mathbf{E}$ embeds the token ids into the residual stream; $L$ causal `TransformerBlock` layers read from and write to the stream; a final normalization and the transposed table $\mathbf{E}^\top$ turn each position's vector into logits over the vocabulary. Left, GPT-2 adds learned position vectors once at the bottom; right, the modern configuration leaves the stream alone at the bottom and rotates queries and keys inside each attention layer via RoPE.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-transformers-gpt-stack.svg)

Inside each block, `CausalAttention` fuses the query, key, and value projections, supports **grouped-query attention** (`num_kv_heads` $\le$ `num_heads`, that section), optionally rotates queries and keys via **RoPE** (`rope=True`, that section), and applies the causal mask so position $t$ attends only to positions $s \le t$:

In [ ]:
class CausalAttention(nn.Module):
    """Fused causal self-attention, optionally rotary; num_kv_heads <
    num_heads shares each key-value head across a group of queries."""
    def __init__(self, num_hiddens, num_heads, *, bias=False, rope=False,
                 num_kv_heads=None):
        super().__init__()
        self.num_heads, self.rope = num_heads, rope
        self.num_kv_heads = num_kv_heads or num_heads
        self.kv_hiddens = self.num_kv_heads * (num_hiddens // num_heads)
        self.W_qkv = nn.Linear(num_hiddens, num_hiddens + 2 * self.kv_hiddens,
                               bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def qkv(self, X, offset=0, tables=None):
        """Queries, keys, and values of X, each (batch, heads, length, head
        width), at positions offset, offset + 1, ...; tables, if given, are
        the d2l.rope_tables of these positions."""
        B, T, D = X.shape
        q, k, v = self.W_qkv(X).split([D, self.kv_hiddens, self.kv_hiddens],
                                      -1)
        q = q.reshape(B, T, self.num_heads, -1).transpose(1, 2)
        k, v = (u.reshape(B, T, self.num_kv_heads, -1).transpose(1, 2)
                for u in (k, v))
        if self.rope:
            q, k = (d2l.rope(u, offset, tables=tables) for u in (q, k))
        return q, k, v

    def forward(self, X, keys=None, values=None, mask=None, tables=None):
        # Self-attention: X also serves as the keys and the values, and the
        # mask is causal, so a block's mask argument must stay None
        if mask is not None:
            raise ValueError('CausalAttention applies its own mask')
        q, k, v = self.qkv(X, tables=tables)
        Y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            enable_gqa=self.num_kv_heads < self.num_heads)
        return self.W_o(Y.transpose(1, 2).flatten(2))  # (batch, length, d)

Our `GPT` class stacks `num_blks` configurable `TransformerBlock` modules (`norm='rms'`, `act='swiglu'`, `pos='rope'`, `pre_norm=True` by default, while also supporting GPT-2's `norm='layer'`, `act='gelu'`, `pos='learned'`, `bias=True`) and ties the output projection to `token_emb.weight`:

In [ ]:
class GPT(nn.Module):
    """Decoder-only transformer language model built from configurable
    blocks."""
    CausalAttention = CausalAttention  # A subclass may replace it

    def __init__(self, vocab_size, *, num_hiddens=256, num_heads=8,
                 num_blks=6, max_len=1024, pos='rope', norm='rms',
                 act='swiglu', pre_norm=True, bias=False, dropout=0,
                 num_kv_heads=None, ffn_factory=None):
        super().__init__()
        assert pos in ('rope', 'learned'), f'unknown pos: {pos!r}'
        self.pos, self.max_len, self.num_hiddens = pos, max_len, num_hiddens
        self.head_width = num_hiddens // num_heads
        self.token_emb = nn.Embedding(vocab_size, num_hiddens)
        nn.init.normal_(self.token_emb.weight, std=0.02)
        if pos == 'learned':
            self.pos_emb = nn.Embedding(max_len, num_hiddens)
            nn.init.normal_(self.pos_emb.weight, std=0.02)
        attn = lambda: self.CausalAttention(
            num_hiddens, num_heads, bias=bias, rope=pos == 'rope',
            num_kv_heads=num_kv_heads)
        self.blks = nn.ModuleList([
            d2l.TransformerBlock(num_hiddens, num_heads, dropout=dropout,
                                 norm=norm, act=act, pre_norm=pre_norm,
                                 bias=bias, attn_factory=attn,
                                 ffn_factory=ffn_factory)
            for _ in range(num_blks)])
        self.norm = (nn.RMSNorm if norm == 'rms'
                     else nn.LayerNorm)(num_hiddens, eps=1e-5)
        self.init_weights()

    def forward(self, X):                     # X: (batch, T) token ids
        H, tables = self.token_emb(X), None   # (batch, T, d)
        if self.pos == 'learned':
            H = H + self.pos_emb(torch.arange(X.shape[1], device=X.device))
        else:  # RoPE angles of positions 0, ..., T - 1, for every block
            tables = d2l.rope_tables(X.shape[1], self.head_width,
                                     device=X.device)
        for blk in self.blks:                 # L blocks, causal attention
            H = blk(H, tables=tables)
        return F.linear(self.norm(H), self.token_emb.weight)  # (batch, T, V)

Following GPT-2 [@Radford.Wu.Child.ea.2019] and that section, `init_weights` draws linear and embedding weights from $\mathcal{N}(0, 0.02^2)$ while scaling the two residual output projections of each block (`attention.W_o` and `ffn.W_2`) by $1 / \sqrt{2L}$:

In [ ]:
@d2l.add_to_class(GPT)
def init_weights(self):
    """GPT-2's initialization of the linear layers: N(0, 0.02^2) weights and
    zero biases; W_o and W_2, which write to the stream, get std
    0.02 / sqrt(2L)."""
    L = len(self.blks)
    for name, m in self.named_modules():
        if isinstance(m, nn.Linear):
            std = 0.02 / (2 * L) ** 0.5 if name.endswith(
                ('W_o', 'W_2')) else 0.02
            nn.init.normal_(m.weight, std=std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

## Training a Byte-Level GPT on WikiText-103

To keep our from-scratch experiments self-contained and free of tokenizer artifacts, we train directly on raw UTF-8 bytes ($V = 256$) from **WikiText-103** [@Merity.Xiong.Bradbury.ea.2016] using mini-batches of $B = 64$ windows of length $T = 128$ bytes ($8{,}192$ tokens per step):

In [ ]:
data = d2l.WikiText103(batch_size=64, num_steps=128)
print(f'{len(data.train) / 1e6:.0f}M training bytes, '
      f'{len(data.val) / 1e6:.2f}M validation bytes, '
      f'vocabulary of {len(data.vocab)}')

In [ ]:
torch.set_float32_matmul_precision('high')  # fp32 matmuls may use tf32

def gpt(*args, **kwargs):  # A fresh model, on the GPU, compiled
    return torch.compile(GPT(*args, **kwargs).to(d2l.try_gpu()))

### The Autoregressive Objective

Given an input window $\mathbf{x} = (x_1, \dots, x_T)$ and shifted target window $\mathbf{y} = (x_2, \dots, x_{T+1})$, the causal mask ensures that row $t$ of the output logits $\mathbf{z}_t \in \mathbb{R}^V$ depends only on $(x_1, \dots, x_t)$. Training minimizes the mean cross-entropy loss across all $T$ positions in parallel:

$$
\mathcal{L}(\boldsymbol{\theta}) = -\frac{1}{T}\sum_{t=1}^{T} \log \textrm{softmax}(\mathbf{z}_t)_{x_{t+1}}.
$$

At initialization, before any learning occurs, the model assigns roughly uniform probability $1/V$ to every byte, so the initial cross-entropy loss is close to $\ln V = \ln 256 \approx 5.55$ nats/byte:

In [ ]:
start = data.train.tobytes().find(b'the European lobster')
X, Y = data.windows(data.train, [start])  # One window and its targets
print('X[0]:', repr(data.vocab.decode(X[0, :24].tolist())))
print('Y[0]:', repr(data.vocab.decode(Y[0, :24].tolist())))
print(np.asarray(d2l.causal_mask(5)).astype(int))

In [ ]:
model = gpt(len(data.vocab), max_len=128)
loss = math.log(d2l.lm_perplexity(model, [next(data.val_dataloader())]))
print(f'{d2l.num_params(model) / 1e6:.2f}M parameters; loss before '
      f'training {loss:.2f} nats per byte, log 256 = {math.log(256):.2f}')

### The Training Recipe (`LMOptimizer`)

Modern LLM pretraining pairs **AdamW** ($\beta_1 = 0.9, \beta_2 = 0.95$, that section) with **selective weight decay** (applying weight decay $\lambda = 0.1$ only to 2D linear projection matrices while exempting embeddings, biases, and 1D normalization scales), **linear warmup followed by cosine learning-rate decay** (that section), and **global gradient-norm clipping** ($\|\nabla \mathcal{L}\|_2 \le 1.0$):

In [ ]:
class LMOptimizer:
    """AdamW with weight decay on the weight matrices only, a linear warmup
    and a cosine decay of the learning rate, and gradient-norm clipping."""
    def __init__(self, model, lr, num_steps, warmup, weight_decay=0.1,
                 max_norm=1.0):
        named = list(model.named_parameters())
        # Embedding tables are named *emb* (token_emb, pos_emb) by convention
        decay = [p for n, p in named if p.ndim == 2 and 'emb' not in n]
        rest = [p for n, p in named if p.ndim != 2 or 'emb' in n]
        self.optimizer = torch.optim.AdamW(
            [{'params': decay, 'weight_decay': weight_decay},
             {'params': rest, 'weight_decay': 0.0}], lr=lr)
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(
            self.optimizer, lambda t: d2l.cosine_warmup(t, num_steps, warmup))
        self.params, self.max_norm = decay + rest, max_norm

    def zero_grad(self):
        self.optimizer.zero_grad()

    def step(self):
        nn.utils.clip_grad_norm_(self.params, self.max_norm)
        self.optimizer.step()
        self.scheduler.step()

Training our 6-block byte-level `GPT` ($d = 256, H = 8, L = 6$, $\approx 4.8\textrm{M}$ parameters) for 1,000 steps ($\approx 8.2\textrm{M}$ bytes) drives validation cross-entropy well below both the empirical unigram entropy ($3.17$ nats/byte) and the empirical bigram entropy ($2.45$ nats/byte) of English Wikipedia text:

In [ ]:
model = gpt(len(data.vocab), max_len=128)
num_steps, every = 1000, 50
optimizer = LMOptimizer(model, lr=2e-3, num_steps=num_steps, warmup=100)
steps, train_curve, val_curve, timer = [], [], [], d2l.Timer()
for step in range(every, num_steps + 1, every):
    timer.start()
    losses = d2l.train_steps(model, data, optimizer, every)
    timer.stop()  # Training time only, without the validation passes
    steps.append(step)
    train_curve.append(sum(losses) / every)
    val_curve.append(math.log(d2l.lm_perplexity(model, data.val_dataloader())))
d2l.plot(steps, [train_curve, val_curve], 'step', 'loss (nats)',
         legend=['train', 'validation'], fmts=('-', '--'))
print(f'step {num_steps}: train {train_curve[-1]:.2f}, '
      f'validation {val_curve[-1]:.2f} nats per byte; '
      f'{timer.sum():.0f} s of training')

In [ ]:
text = data.train[:10**7].astype(np.int64)  # The first 10M training bytes
V = len(data.vocab)
unigram = np.bincount(text, minlength=V) / len(text)
pairs = np.bincount(text[:-1] * V + text[1:], minlength=V * V).reshape(V, V)
bigram = pairs / np.maximum(pairs.sum(1, keepdims=True), 1)
p = unigram[unigram > 0]
print(f'unigram {-(p * np.log(p)).sum():.2f}, '
      f'bigram {-np.log(bigram[text[:-1], text[1:]]).mean():.2f} '
      f'nats per byte')

## Controlled Comparisons

### Run-to-Run Spread and FFN Activation (`SwiGLU` vs. `GELU`)

To separate genuine architectural improvements from random seed noise, we train three independent seeds of our default `SwiGLU` model and three seeds of a parameter-matched `GELU` model (`act='gelu'`):

In [ ]:
def train_lm(model, data, lr=2e-3, num_steps=1000):
    """Train with the recipe of the main run; return milliseconds per step."""
    optimizer = LMOptimizer(model, lr=lr, num_steps=num_steps, warmup=100)
    # One untimed call of 50 steps warms up (JAX compiles), the rest is timed
    bench = d2l.Benchmark(lambda: d2l.train_steps(model, data, optimizer, 50),
                          warmup=1, repeats=num_steps // 50 - 1)
    return 1000 * bench.time / 50

runs = {'swiglu': [val_curve[-1]], 'gelu': []}  # The main run comes first
times = {'swiglu': [], 'gelu': []}  # Milliseconds per step
for act, repeats in (('swiglu', 2), ('gelu', 3)):
    for _ in range(repeats):
        net = gpt(len(data.vocab), act=act, max_len=128)
        times[act].append(train_lm(net, data))
        runs[act].append(math.log(d2l.lm_perplexity(net,
                                                    data.val_dataloader())))
    print(f'{act:>6} ({d2l.num_params(net):,} parameters): validation '
          + ', '.join(f'{v:.3f}' for v in runs[act])
          + f'; mean {np.mean(runs[act]):.3f}, range {np.ptp(runs[act]):.3f}')

Run-to-run variation across random seeds is only $\sim 0.01$ nats/byte, whereas `SwiGLU` consistently beats `GELU` at the same parameter and FLOP budget by $\sim 0.06$ nats/byte. (Similarly, switching `pre_norm=True` to `pre_norm=False` without warmup causes training to stall at the unigram loss of $3.17$ nats/byte due to the early-layer gradient attenuation and token collapse shown in that section, and even with warmup post-norm trails pre-norm by a few hundredths of a nat per byte.)

### Comparing Against a Matched Recurrent Baseline

How does our 6-block `GPT` compare against the recurrent language model (`d2l.RNNLM`, that section) given the **same parameter count** ($d_{\textrm{rnn}} = 1920$, $\approx 4.7\textrm{M}$ parameters) and training steps? We sweep the RNN learning rate across $\{10^{-3}, 2\times 10^{-3}, 4\times 10^{-3}\}$ and compare validation loss as a function of window position $t \in \{1, \dots, 128\}$ (`loss_by_position`):

In [ ]:
def rnnlm(vocab_size, num_hiddens):  # The tanh RNN LM of the Basics
    return d2l.RNNLM(d2l.RNN(256, num_hiddens), vocab_size).to(d2l.try_gpu())

@torch.inference_mode()
def loss_by_position(model, data):
    """Mean validation loss in nats at each position of the window."""
    device = next(model.parameters()).device
    total, count = 0, 0
    for X, Y in data.val_dataloader():
        logits = model(X.to(device))  # (batch, T, V)
        total += F.cross_entropy(logits.transpose(1, 2), Y.to(device),
                                 reduction='none').sum(0)
        count += len(X)
    return (total / count).tolist()

In [ ]:
rnn_runs = []  # (validation loss, model, ms per step) of every run
for lr in (1e-3, 2e-3, 4e-3):
    for _ in range(2):
        rnn = rnnlm(len(data.vocab), 1920)
        ms = train_lm(rnn, data, lr=lr)
        rnn_runs.append((np.mean(loss_by_position(rnn, data)), rnn, ms))
    print(f'RNN ({d2l.num_params(rnn):,} parameters), peak rate {lr:g}: '
          'validation ' + ', '.join(f'{r[0]:.3f}' for r in rnn_runs[-2:]))
best = min(rnn_runs, key=lambda run: run[0])
positions = (0, 1, 4, 16, 64, 127)
print('validation loss at positions', ', '.join(map(str, positions)))
for name, net, ms in (('GPT', model, np.mean(times['swiglu'])),
                      ('best RNN', *best[1:])):
    loss = loss_by_position(net, data)
    print(f'{name:>8}: ' + ' '.join(f'{loss[t]:.2f}' for t in positions)
          + f'; {ms:.1f} ms per training step')

At position $t = 1$ (where only a single byte of context is available), `GPT` and `RNNLM` start at nearly the same loss. As more prefix context becomes available ($t \to 128$), the Transformer's validation loss drops more steeply than the RNN's because self-attention reads any prior byte in $O(1)$ hops rather than compressing history through $t$ sequential recurrent steps.

## Sampling from the Model

Autoregressive generation (`GPT.generate`) appends one token at a time: at each step it feeds the current prefix through the model, divides the final position's logits $\mathbf{z}_t$ by a **temperature** $\tau > 0$, optionally masks all but the **top-$k$** highest logits to $-\infty$ [@Fan.Lewis.Dauphin.2018], and samples the next token from $\textrm{softmax}(\mathbf{z}_t / \tau)$ (or takes $\arg\max$ when $\tau = 0$):

In [ ]:
@d2l.add_to_class(GPT)
@torch.inference_mode()
def generate(self, prefix, num_tokens, temperature=1.0, top_k=None):
    """Sample a continuation of the token-id list prefix."""
    ids = torch.tensor(prefix, device=next(self.parameters()).device)
    for _ in range(num_tokens):  # The tokens stay on the device
        logits = self(ids[None, -self.max_len:])[0, -1] / temperature
        if top_k is not None:
            cutoff = torch.topk(logits, top_k).values[-1]
            logits[logits < cutoff] = -torch.inf
        ids = torch.cat((ids, torch.multinomial(F.softmax(logits, -1), 1)))
    return ids.tolist()

In [ ]:
prefix = data.vocab.encode('The history of the ')
for temperature, top_k in ((1.0, None), (0.7, 8), (2.0, None)):
    out = model.generate(prefix, 120, temperature=temperature, top_k=top_k)
    d2l.print_wrapped(f'T={temperature}, top_k={top_k}: '
                      + repr(data.vocab.decode(out)))

## Loading Pretrained GPT-2 (`124M`)

Because our `GPT` class supports `norm='layer'`, `act='gelu'`, `pos='learned'`, and `bias=True`, it can instantiate OpenAI's 124M-parameter **GPT-2 Small** architecture ($V = 50{,}257, d = 768, H = 12, L = 12, T_{\max} = 1024$) and load the official `model.safetensors` weights and byte-level BPE tokenizer (`gpt2_tokenizer`) directly:

In [ ]:

GPT2_URL = 'https://huggingface.co/openai-community/gpt2/resolve/main/'
d2l.DATA_HUB['gpt2-weights'] = (
    GPT2_URL + 'model.safetensors',
    '89a76996d7c6ee89b86618a265483aab73e61d50')
d2l.DATA_HUB['gpt2-merges'] = (
    GPT2_URL + 'merges.txt', '396d4d8ec90cb02f4d56e049e0e4add868bcd943')
d2l.DATA_HUB['gpt2-encoder'] = (
    GPT2_URL + 'vocab.json', 'f0223209235343bc067d7da838328bced8085ae1')

def gpt2_tokenizer():
    """GPT-2's byte-level BPE tokenizer, assembled by tiktoken."""
    import tiktoken  # lazy: d2l itself does not require tiktoken
    from tiktoken.load import data_gym_to_mergeable_bpe_ranks
    return tiktoken.Encoding(
        'gpt2', explicit_n_vocab=50257,
        pat_str=d2l.BPETokenizer.GPT2_PATTERN,
        mergeable_ranks=data_gym_to_mergeable_bpe_ranks(
            d2l.download('gpt2-merges', '../data/gpt2'),
            d2l.download('gpt2-encoder', '../data/gpt2')),
        special_tokens={'<|endoftext|>': 50256})

enc = gpt2_tokenizer()
ids = enc.encode('Attention is all you need.')
print(ids, [enc.decode([i]) for i in ids])

<!-- tbl-caption:Mapping between OpenAI's GPT-2 `safetensors` checkpoint keys and our `GPT` module attributes (`Conv1D` weights have shape `(in, out)` and are transposed on load in PyTorch). -->

| Checkpoint tensor | `GPT` attribute | Note |
|:--|:--|:--|
| `wte.weight` | `token_emb.weight` | Shared with output head (`V x d`) |
| `wpe.weight` | `pos_emb.weight` | 1,024 learned position vectors |
| `h.i.ln_1`, `h.i.ln_2` | `blks[i].norm1`, `blks[i].norm2` | Pre-attention and pre-FFN LayerNorms |
| `h.i.attn.c_attn` | `blks[i].attention.W_qkv` | Fused QKV (`Conv1D`, transpose in PyTorch) |
| `h.i.attn.c_proj` | `blks[i].attention.W_o` | Output projection (`Conv1D`) |
| `h.i.mlp.c_fc`, `h.i.mlp.c_proj` | `blks[i].ffn.W_1`, `blks[i].ffn.W_2` | Two-layer GELU MLP (`Conv1D`) |
| `ln_f` | `norm` | Final LayerNorm before output head |

In [ ]:
def load_gpt2():
    """GPT-2 small (124M) as a GPT with the released weights, in eval
    mode."""
    from safetensors.torch import load_file  # lazy, like tiktoken above
    gpt2 = GPT(vocab_size=50257, num_hiddens=768, num_heads=12, num_blks=12,
               max_len=1024, pos='learned', norm='layer', act='gelu',
               pre_norm=True, bias=True)
    weights = load_file(d2l.download('gpt2-weights', '../data/gpt2'))
    with torch.no_grad():
        gpt2.token_emb.weight.copy_(weights['wte.weight'])
        gpt2.pos_emb.weight.copy_(weights['wpe.weight'])
        gpt2.norm.weight.copy_(weights['ln_f.weight'])
        gpt2.norm.bias.copy_(weights['ln_f.bias'])
        for i, blk in enumerate(gpt2.blks):
            modules = {f'h.{i}.ln_1': blk.norm1, f'h.{i}.ln_2': blk.norm2,
                       f'h.{i}.attn.c_attn': blk.attention.W_qkv,
                       f'h.{i}.attn.c_proj': blk.attention.W_o,
                       f'h.{i}.mlp.c_fc': blk.ffn.W_1,
                       f'h.{i}.mlp.c_proj': blk.ffn.W_2}
            for key, module in modules.items():
                W = weights[key + '.weight']  # Conv1D: (in, out)
                module.weight.copy_(W.T if W.ndim == 2 else W)
                module.bias.copy_(weights[key + '.bias'])
    return gpt2.eval()

gpt2 = load_gpt2().to(d2l.try_gpu())
print(f'{d2l.num_params(gpt2) / 1e6:.1f}M parameters')

We verify the loaded checkpoint by computing its cross-entropy on the opening paragraph of *The Time Machine* ($\approx 3.9$ nats/token, or perplexity $\approx 49$) and generating continuations:

In [ ]:
text = ("The Time Machine, by H. G. Wells. The Time Traveller was "
        "expounding a recondite matter to us.")
x = torch.tensor(enc.encode(text), device=d2l.try_gpu())[None]
with torch.inference_mode():
    logits = gpt2(x)
loss = F.cross_entropy(logits[0, :-1], x[0, 1:])
print(f'per-token loss {loss.item():.2f}, '
      f'perplexity {loss.exp().item():.0f}')

In [ ]:
out = gpt2.generate(enc.encode('Alan Turing theorized that computers '
                               'would one day become'), 16, top_k=1)
d2l.print_wrapped(enc.decode(out))

In [ ]:
out = gpt2.generate(enc.encode('The secret of a good deep learning '
                               'textbook is'), 40, temperature=0.8,
                    top_k=50)
d2l.print_wrapped(enc.decode(out))

## The Modern Recipe

Comparing the 2019 `load_gpt2()` configuration against our default `GPT` highlights the four core upgrades that define modern open-weights decoder-only language models (the table):

1. **Normalization:** Pre-norm `RMSNorm` without biases (that section) instead of `LayerNorm`, often paired with per-head **QK-Norm** (that section) or sublayer-output normalization (OLMo 2, Gemma 3).
2. **Positions:** **Rotary Position Embeddings (RoPE)** (that section) inside each attention layer instead of a fixed learned position table `wpe`.
3. **Feed-forward network:** Bias-free gated **`SwiGLU`** (or `GeGLU`) instead of a two-layer `GELU` MLP.
4. **Inference cache & capacity scaling:** **Grouped-query attention (GQA)**, **sliding-window attention**, or **low-rank latent compression (MLA)** to shrink the KV cache (that section), and **Mixture-of-Experts (MoE)** layers (that section) to scale total parameters without scaling per-token FLOPs.

<!-- tbl-caption:Architectures of seven open-weights model families, as reported in their technical reports, 2023 to 2025. Attention and cache: that section. Normalization: that section. Positions: that section. Mixture of experts: that section. -->

| Model | Attention and cache | Normalization | Positions | FFN | Dense or MoE |
|:--|:--|:--|:--|:--|:--|
| Mistral 7B (2023) [@Jiang.Sablayrolles.Mensch.ea.2023] | GQA $32{:}8$; sliding window 4096 | RMSNorm, pre | RoPE, base $10^4$ | SwiGLU | Dense |
| Llama 3 (2024) | GQA $32{:}8$ (8B), $64{:}8$ (70B), $128{:}8$ (405B) | RMSNorm, pre | RoPE, base $5 \times 10^5$ | SwiGLU | Dense |
| Qwen3 (2025) [@Yang.Li.Yang.ea.2025] | GQA: $64{:}8$ dense (32B), $64{:}4$ MoE (235B-A22B) | RMSNorm, pre; QK-norm | RoPE, base $10^6$; YaRN | SwiGLU | Dense, and MoE: 128 experts, 8 active |
| OLMo 2/3 (2024/25) [@OLMo.2025; @OLMo3.2025] | MHA (7B), GQA (32B); OLMo 3: window 4096 on 3 of 4 layers | RMSNorm on sublayer outputs; QK-norm | RoPE | SwiGLU | Dense |
| DeepSeek-V3 (2024) [@Liu.Feng.Xue.ea.2024] | MLA: KV compressed to a 512-dim latent | RMSNorm, pre | Decoupled RoPE (64-dim shared key) | SwiGLU | MoE: 256 experts + 1 shared, 8 active |
| Gemma 3 (2025) [@Gemma.Team.2025] | GQA; local:global $5{:}1$, window 1024 | RMSNorm, pre and post; QK-norm | RoPE, base $10^6$ global, $10^4$ local | GeGLU | Dense |
| gpt-oss (2025) [@OpenAI.2025] | GQA $64{:}8$; window 128 on alternate layers; learned sinks | RMSNorm, pre | RoPE; YaRN | SwiGLU, clamped | MoE: 128 or 32 experts, 4 active |

## Summary

* A **decoder-only GPT** wraps $L$ causal `TransformerBlock` layers between a token embedding table $\mathbf{E} \in \mathbb{R}^{V \times d}$ and a final normalization + weight-tied output projection $\mathbf{E}^\top$, predicting next-token logits at all $T$ sequence positions in one parallel forward pass (the equation).
* Training with **AdamW** (selective weight decay on 2D matrices only), linear warmup + cosine decay, and gradient clipping (`LMOptimizer`) drives our 4.8M-parameter byte-level `GPT` well below the bigram entropy of English text in 1,000 steps.
* In controlled comparisons at matched parameters and compute, `SwiGLU` outperforms `GELU` by $\sim 0.06$ nats/byte (versus a $\sim 0.01$-nat seed spread), and `GPT` achieves a steadily widening advantage over a matched `RNNLM` at deeper context positions $t$.
* Modern open-weights LLMs (the table) share the pre-norm `RMSNorm` + `RoPE` + `SwiGLU`/`GeGLU` recipe of our `GPT` class, differing primarily in how they shrink the KV cache (that section) and whether they replace the dense FFN with a Mixture of Experts (that section).

## Exercises

1. **Ablating the training recipe (`LMOptimizer`).** `LMOptimizer` combines four ingredients: selective weight decay ($\lambda = 0.1$ on 2D weight matrices only), linear warmup (100 steps), cosine decay, and global gradient clipping ($\|\nabla \mathcal{L}\|_2 \le 1.0$). Train our 6-block byte-level `GPT` for 1,000 steps while disabling one ingredient at a time: (a) applying weight decay to *all* parameters including embeddings and RMSNorm gains, (b) no warmup (`warmup=0`), (c) constant learning rate (`lr=2e-3` throughout), and (d) no gradient clipping. Which single change hurts validation loss the most?

2. **Auditing the GPT-2 checkpoint tensor map.** The GPT-2 `model.safetensors` file holds 148 weight/bias tensors (plus 12 legacy causal mask buffers `h.i.attn.bias`). Write a short check verifying that `sum(p.numel() for p in gpt2.parameters())` equals the total element count of the 148 parameter tensors in the file ($124{,}439{,}808$), and explain why transposing `c_attn.weight` is required in PyTorch (`nn.Linear`) but not in Flax (`nnx.Linear`).

3. **Zero-layer tied-embedding baseline ($L = 0$).** Consider `GPT` with `num_blks=0`: input token $x_t$ is embedded as $\mathbf{e}_{x_t}$, normalized by `RMSNorm`, and dotted with $\mathbf{E}^\top$ to produce logits $\mathbf{z}_t \propto \mathbf{e}_{x_t} \mathbf{E}^\top$.
   1. What class of statistical language model ($n$-gram order) can a 0-block model represent?
   2. Why does tying the input and output embeddings ($\mathbf{W}_{\textrm{LM}} = \mathbf{E}^\top$) force the symmetric matrix $\mathbf{E}\mathbf{E}^\top$ to assign its largest diagonal entry to $x_{t+1} = x_t$ if all rows of $\mathbf{E}$ have equal norm, and how does the untied or 1-block model break this symmetry?

4. **Comparing bits-per-byte across tokenizers (Byte-level GPT vs. BPE GPT-2).** A byte-level model reports cross-entropy in nats/byte, whereas GPT-2 reports cross-entropy in nats/subword-token.
   1. Show that on any passage of $N_B$ UTF-8 bytes that `enc.encode` tokenizes into $N_T$ subword tokens, the exact **bits-per-byte (bpb)** of a subword model with mean per-token negative log-likelihood $\mathcal{L}_{\textrm{tok}}$ (in nats/token) is $\textrm{bpb} = \frac{N_T \, \mathcal{L}_{\textrm{tok}}}{N_B \ln 2}$.
   2. Evaluate both our 4.8M-parameter byte-level `model` and pretrained 124M-parameter `gpt2` in bits-per-byte on the first 10,000 bytes of the WikiText-103 validation set.

5. **Top-$p$ (nucleus) sampling.** Extend `GPT.generate` with an optional `top_p` argument ($0 < p < 1$) [@Holtzman.Buys.Du.ea.2020] that sorts the temperature-scaled probabilities in descending order and masks out tokens beyond the smallest prefix whose cumulative probability exceeds $p$. Compare completions from `gpt2` on the prompt `"Alan Turing theorized that computers would"` using `top_p=0.9, temperature=0.8` versus unfiltered `temperature=1.2`.

6. **Learning-rate stability with QK-Norm.** Train our 6-block byte-level `GPT` across peak learning rates $\eta \in \{2\times 10^{-3}, 6\times 10^{-3}, 1.5\times 10^{-2}\}$ with standard `CausalAttention` versus a variant that applies `RMSNorm(d_h)` to `q` and `k` inside `CausalAttention.qkv` (that section). How does QK-Norm shift the maximum stable learning rate?

[Discussions](https://d2l.discourse.group/t/36775)